# درس ۱۰ - عامل‌های هوش مصنوعی در محیط تولید

در این درس شما **الگوهای تولید** برای عامل‌های هوش مصنوعی با استفاده از Microsoft Agent Framework (Python) را خواهید آموخت. ما پوشش می‌دهیم:

- **قابلیت مشاهده** — افزودن اندازه‌گیری زمان و ثبت لاگ به تعاملات عامل
- **ارزیابی** — استفاده از یک عامل ارزیاب برای امتیازدهی به کیفیت پاسخ
- **مدیریت هزینه** — استراتژی‌هایی برای بهینه‌سازی توکن‌ها و انتخاب مدل

سناریو یک **عامل مسافرتی** است که به کاربران در برنامه‌ریزی سفرها کمک می‌کند، با نظارت و ارزیابی که در بالای آن لایه‌بندی شده‌اند.


## راه‌اندازی


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## ملاحظات تولید

انتقال عامل‌های هوش مصنوعی از نمونه‌های اولیه به محیط تولید نیازمند توجه دقیق به سه رکن است:

1. **قابلیت مشاهده** — لازم است بتوانید مشاهده کنید که عامل چه کاری انجام می‌دهد، چقدر طول می‌کشد و چه ابزارهایی را فراخوانی می‌کند. بدون ردیابی و لاگ‌گیری، عیب‌یابی مشکلات در محیط تولید تقریباً غیرممکن است.

2. **ارزیابی** — بررسی‌های خودکار کیفیت اطمینان می‌دهند که پاسخ‌های عامل در طول زمان دقیق، کامل و مفید باقی بمانند. یک عامل ارزیابی‌کننده می‌تواند پاسخ‌ها را بر اساس معیارهای تعریف‌شده امتیازدهی کند.

3. **مدیریت هزینه** — مصرف توکن مستقیماً بر هزینه تأثیر می‌گذارد. استراتژی‌هایی مانند بهینه‌سازی پرامپت، انتخاب مدل و استفاده از حافظه پنهان کمک می‌کنند هزینه‌ها را بدون فدا کردن کیفیت تحت کنترل نگه دارید.


## ساخت یک عامل قابل مشاهده

ما ابزارهای سفر را تعریف می‌کنیم و فراخوانی عامل را با زمان‌سنجی می‌پوشانیم تا بتوانیم تأخیر را پایش کنیم. در محیط تولید شما آن را با OpenTelemetry یا یک سامانه ردیابی مشابه یکپارچه خواهید کرد.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## الگوهای ارزیابی

یک الگوی رایج در محیط تولید استفاده از یک عامل دوم به‌عنوان **ارزیاب** است. ارزیاب پاسخ عامل اصلی را بر اساس معیارهای از پیش تعریف‌شده‌ای مانند کامل بودن، دقت و مفید بودن امتیازدهی می‌کند.

این امکان را فراهم می‌آورد:
- دروازه‌های کیفیت خودکار قبل از رسیدن پاسخ‌ها به کاربران
- کشف رگرسیون زمانی که پرامپت‌ها یا مدل‌ها تغییر می‌کنند
- پایش مداوم عملکرد عامل در طول زمان


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## استراتژی‌های مدیریت هزینه

کنترل هزینه‌ها برای عامل‌های تولیدی هوش مصنوعی حیاتی است. در اینجا استراتژی‌های کلیدی آمده‌اند:

| استراتژی | توضیحات |
|---|---|
| **بهینه‌سازی پرامپت** | دستورالعمل‌های سیستمی را مختصر نگه دارید. زمینه‌های زائد را حذف کنید تا تعداد توکن‌های ورودی کاهش یابد. |
| **انتخاب مدل** | از مدل‌های کوچک‌تر و ارزان‌تر (مثلاً GPT-4o-mini) برای کارهای ساده مانند طبقه‌بندی یا استخراج استفاده کنید، و مدل‌های بزرگ‌تر را برای استدلال پیچیده‌تر نگه دارید. |
| **ذخیرهٔ موقت** | نتایج ابزار و پرس‌وجوهای پرتکرار را در کش ذخیره کنید تا از فراخوانی‌های تکراری API جلوگیری شود. |
| **بودجه توکن‌ها** | محدودیت‌های `max_tokens` را تنظیم کنید تا از پاسخ‌های غیرمنتظره طولانی جلوگیری شود. |
| **پردازش دسته‌ای** | در صورت امکان چند پرسش کاربر را در یک فراخوانی API گروه‌بندی کنید. |

در عمل، رویکرد چندسطحی خوب عمل می‌کند: درخواست‌های ساده را به مدلی سریع و کم‌هزینه هدایت کنید و تنها درخواست‌های پیچیده را به مدلی توانمندتر ارجاع دهید.


## خلاصه

در این درس یاد گرفتید چگونه:

1. **افزودن قابلیت مشاهده‌پذیری** به تعاملات عامل با زمان‌بندی و ثبت لاگ، که زمینه را برای ردیابی و نظارت فراهم می‌کند.
2. **ارزیابی پاسخ‌های عامل** به‌صورت خودکار با استفاده از یک عامل ارزیاب که کامل بودن، دقت و مفید بودن را امتیازدهی می‌کند.
3. **مدیریت هزینه‌ها** از طریق بهینه‌سازی پرامپت، انتخاب مدل، ذخیره‌سازی در کش، و بودجه‌بندی توکن‌ها.

این الگوهای تولیدی به اطمینان از اینکه عامل‌های هوش مصنوعی شما قابل‌اعتماد، قابل‌اندازه‌گیری و از نظر هزینه در مقیاس به‌صرفه باشند کمک می‌کنند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
سلب مسئولیت:
این سند با استفاده از سرویس ترجمهٔ هوش مصنوعی Co‑op Translator (https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است حاوی خطاها یا نواقصی باشند. نسخهٔ اصلی سند به زبان مبدأ باید به‌عنوان مرجع معتبر در نظر گرفته شود. برای اطلاعات حساس یا حیاتی، ترجمهٔ حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوتفاهم یا تفسیر نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
